In [ ]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 小写，保留字母和空格，去除标点
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. 按空格分词
    tokens = text.split()

    # 3. 按频率排序构建词汇表
    freq = Counter(tokens)
    sorted_words = sorted(freq.keys(), key=lambda w: (-freq[w], w))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(tokens) - n + 1):
        window = tokens[i:i+n]
        features.append(window)
        if i + n < len(tokens):
            labels.append(tokens[i+n])
        else:
            labels.append(None)   

    return vocab, (features, labels)

text = "The time machine"
n = 2
vocab, (features, labels) = preprocess_text(text, n)
print(vocab)        
print(features)      
print(labels)         

In [ ]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    x_t:   (batch_size, input_size)
    h_prev:(batch_size, hidden_size)
    W_hx:  (input_size, hidden_size)
    W_hh:  (hidden_size, hidden_size)
    b_h:   (hidden_size,)
    """
    a = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a)
    return h_t, a

def rnn_cell_backward(dh_next, x_t, h_prev, W_hx, W_hh, b_h, a, h_t):

    # tanh 导数
    da = dh_next * (1 - h_t ** 2)   # (batch_size, hidden_size)

    dx_t = da @ W_hx.T              # (batch_size, input_size)
    dh_prev = da @ W_hh.T           # (batch_size, hidden_size)

    dW_hx = x_t.T @ da              # (input_size, hidden_size)
    dW_hh = h_prev.T @ da           # (hidden_size, hidden_size)
    db_h = np.sum(da, axis=0)       # (hidden_size,)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h

In [ ]:
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, hidden_dim, num_layers=1):
  
    seq_len, batch, input_dim = X.shape

    rnn = nn.RNN(
        input_size=input_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        bidirectional=True,
        batch_first=False
    )

    output, h_n = rnn(X)   # output: (seq_len, batch, 2*hidden_dim)
                           # h_n:    (num_layers*2, batch, hidden_dim)

    # 每个时间步拼接后的隐藏状态
    all_states = output

    # 取最后一层的前向和后向最终隐藏状态
    # h_n 排列: [层0前向, 层0反向, 层1前向, 层1反向, ...]
    forward_last = h_n[-2, :, :]   # 最后一层前向
    backward_last = h_n[-1, :, :]  # 最后一层反向
    final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)

    return all_states, final_state

In [ ]:
import torch
import torch.nn.functional as F

def cbow_loss(context_indices, target_indices, W, W_out):
   
    # 取上下文词向量
    embed = W[context_indices]          # (batch, context_size, d)
    hidden = embed.mean(dim=1)          # (batch, d)

    logits = hidden @ W_out             # (batch, V)
    loss = F.cross_entropy(logits, target_indices)

    return loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    """
    Q: (..., seq_len_q, d_k)
    K: (..., seq_len_k, d_k)
    V: (..., seq_len_k, d_v)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    attn_weights = F.softmax(scores, dim=-1)
    return attn_weights @ V

def multi_head_attention_forward(X, W_q, W_k, W_v, W_o, num_heads=2):
    """
    X: (seq_len, batch, d_model), d_model=4, num_heads=2
    W_q, W_k, W_v, W_o: 线性层权重，形状均为 (d_model, d_model)
    返回: (seq_len, batch, d_model)
    """
    seq_len, batch, d_model = X.shape
    d_k = d_v = d_model // num_heads   # 2

    # 线性投影
    Q = X @ W_q   # (seq_len, batch, d_model)
    K = X @ W_k
    V = X @ W_v

    # 拆分为多头
    Q = Q.view(seq_len, batch, num_heads, d_k).transpose(0, 1).transpose(1, 2)
    K = K.view(seq_len, batch, num_heads, d_k).transpose(0, 1).transpose(1, 2)
    V = V.view(seq_len, batch, num_heads, d_v).transpose(0, 1).transpose(1, 2)
    # 形状: (batch, num_heads, seq_len, d_k/d_v)

    # 每个头独立计算注意力
    attn_output = scaled_dot_product_attention(Q, K, V)
    # 形状: (batch, num_heads, seq_len, d_v)

    # 拼接多头
    attn_output = attn_output.transpose(1, 2).contiguous()
    attn_output = attn_output.view(batch, seq_len, d_model)
    # 形状: (batch, seq_len, d_model)

    # 转回 (seq_len, batch, d_model)
    attn_output = attn_output.transpose(0, 1)

    # 最终线性投影
    output = attn_output @ W_o   # (seq_len, batch, d_model)

    return output